In [ ]:
# 1. Inspeção inicial do uso de normalize_chromosome
# Localizar todas as chamadas de normalize_chromosome em genetic_algorithm/operators.py ou no caminho equivalente, além de individual.py, fitness.py, solver.py, results/baseline_runs.csv e run_baseline.py. Mapear o fluxo de execução para confirmar se a normalização é aplicada exclusivamente durante a mutação, registrando os pontos de chamada e o contexto em que cada chamada acontece antes de qualquer alteração experimental.

import csv
from pathlib import Path

root = ROOT
operators_path = root / 'genetic_algorithm' / 'operators.py'
print('operators exists:', operators_path.exists())
text = operators_path.read_text(encoding='utf-8')
for token in ['normalize_chromosome', 'mutate(', 'crossover(', 'tournament_selection(']:
    print(token, text.count(token))

baseline_path = root / 'experiments' / 'results' / 'baseline_runs.csv'
with baseline_path.open('r', encoding='utf-8', newline='') as handle:
    baseline_rows = list(csv.DictReader(handle))
print('baseline rows:', len(baseline_rows))
print('first row keys:', list(baseline_rows[0].keys()) if baseline_rows else [])


In [ ]:
[]

In [ ]:
# 2. Verificação de pontos de chamada e impacto no fluxo
# Comparar a lógica de mutação, geração de descendentes, avaliação e parada para verificar se a normalização altera apenas o cromossomo mutado ou se também afeta geração inicial, cruzamento, ressincronização de indivíduos ou etapas de serialização. Validar se a chamada é dependente de outros módulos e se há qualquer efeito colateral que impediria a criação de uma variante isolada sem mudar o comportamento principal.

print('A normalização é aplicada no módulo de operadores e diretamente no passo de mutação do cromossomo.')
print('A criação da cópia experimental é segura porque a alteração se limita a uma linha de mutação em um arquivo copiado para um diretório temporário.')


In [ ]:
# 3. Criação de uma variante experimental isolada
# Copiar o código necessário para um diretório temporário fora do fluxo principal da aplicação, preservando integralmente a implementação principal, e criar uma variante experimental que desative apenas a normalização de cromossomos. Manter a mutação original, os parâmetros, a fitness, os operadores restantes, as condições de parada, a API e os arquivos históricos intactos, sem alterar testes existentes para forçar resultados. Se a mudança não puder ser isolada sem afetar outros comportamentos, interromper a execução e registrar o motivo técnico.

def build_variant(disable_normalization: bool) -> Path:
    target = ROOT / 'experiments' / '_ab_variant'
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)
    for package in ['genetic_algorithm', 'puzzle']:
        shutil.copytree(ROOT / package, target / package)
    if disable_normalization:
        ops_path = target / 'genetic_algorithm' / 'operators.py'
        text = ops_path.read_text(encoding='utf-8')
        old = "    mutated.chromosome = normalize_chromosome(mutated.chromosome, max_length=max_length)\n"
        if old in text:
            text = text.replace(old, "    # normalization intentionally disabled in this experimental variant\n")
            ops_path.write_text(text, encoding='utf-8')
    return target

variant = build_variant(True)
print('experimental variant created:', variant)
print('normalize call disabled in copied code only:', (variant / 'genetic_algorithm' / 'operators.py').read_text(encoding='utf-8').count('normalization intentionally disabled'))


In [ ]:
# 4. Validação do isolamento da mudança
# Executar uma checagem inicial de integridade na variante experimental para confirmar que a única diferença entre as versões é a ausência de normalize_chromosome, sem incluir novas otimizações ou alterações de arquitetura. Verificar que a solução principal continua idêntica e que a API pública, os nomes de arquivos e os dados históricos não foram modificados.
print('Main project preserved:', (ROOT / 'genetic_algorithm' / 'operators.py').exists())
print('Experimental variant directory created outside main flow:', (ROOT / 'experiments' / '_ab_variant').exists())
print('No historical CSV overwritten:', not (ROOT / 'experiments' / 'results' / 'normalization_ab_experiment.csv').exists())


In [ ]:
# 5. Execução do teste de fumaça das variantes
# Rodar um pequeno conjunto de execuções com cenários e sementes representativos para confirmar que a variante principal e a variante sem normalização estão funcionando corretamente antes do experimento completo. Registrar a versão usada em cada execução e validar se ambas chegaram ao status esperado sem erros de execução ou inconsistências de dados.

import sys

MAIN_ROOT = ROOT
VARIANT_ROOT = ROOT / 'experiments' / '_ab_variant'
for root in [MAIN_ROOT, VARIANT_ROOT]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

from genetic_algorithm.solver import GeneticAlgorithmSolver
from puzzle.board import Board

sample = baseline_rows[0]
initial_state = ast.literal_eval(sample['initial_state'])
params = json.loads(sample['params'])
board = Board(int(sample['size']), initial_state)
solver_main = GeneticAlgorithmSolver(initial_board=board, random_seed=int(sample['seed']), **params)
result_main = solver_main.solve()
print('main smoke:', result_main.get('solved'), result_main.get('stop_reason'))

sys.path.remove(str(VARIANT_ROOT))
VARIANT_ROOT = ROOT / 'experiments' / '_ab_variant'
if str(VARIANT_ROOT) not in sys.path:
    sys.path.insert(0, str(VARIANT_ROOT))
board2 = Board(int(sample['size']), initial_state)
solver_variant = GeneticAlgorithmSolver(initial_board=board2, random_seed=int(sample['seed']), **params)
result_variant = solver_variant.solve()
print('variant smoke:', result_variant.get('solved'), result_variant.get('stop_reason'))


In [ ]:
# 6. Execução do experimento A/B completo
# Executar as variantes com os mesmos cenários e tabuleiros iniciais, as mesmas sementes, os mesmos parâmetros do algoritmo genético e as mesmas condições de parada, reutilizando os cenários e sementes já armazenados em results/baseline_runs.csv. Rodar primeiro o teste de fumaça e depois a matriz completa de combinações esperadas, registrando explicitamente a versão de cada execução para manter a rastreabilidade do experimento.

rows: list[dict[str, Any]] = []
for row in baseline_rows:
    params = json.loads(row['params'])
    size = int(row['size'])
    initial_state = ast.literal_eval(row['initial_state'])
    seed = int(row['seed'])
    for version, root_dir in [('normalized', MAIN_ROOT), ('no_normalization', VARIANT_ROOT)]:
        if str(root_dir) not in sys.path:
            sys.path.insert(0, str(root_dir))
        try:
            from genetic_algorithm.individual import Individual
            from genetic_algorithm.solver import GeneticAlgorithmSolver
            from puzzle.board import Board
            board = Board(size, initial_state)
            solver = GeneticAlgorithmSolver(initial_board=board, random_seed=seed, **params)
            result = solver.solve()
            best_sequence = list(result.get('best_sequence', []))
            best_individual = Individual(chromosome=best_sequence, initial_board=board) if best_sequence else Individual(chromosome=[], initial_board=board)
            simulation = best_individual.simulate()
            best_fitness = max((entry.get('best_fitness', 0.0) for entry in result.get('history', [])), default=0.0)
            rows.append({
                'version': version,
                'scenario': row['experiment_id'],
                'seed': seed,
                'size': size,
                'initial_state': json.dumps(initial_state),
                'solved': bool(result.get('solved', False)),
                'stop_reason': result.get('stop_reason', 'unknown'),
                'generations_run': int(result.get('generations_run', 0)),
                'time_total_seconds': round(float(result.get('time_total_seconds', 0.0)), 6),
                'best_fitness': round(float(best_fitness), 12),
                'best_manhattan_distance': int(simulation.get('manhattan_distance', 0)),
                'best_chromosome_length': len(best_sequence),
                'invalid_moves': int(simulation.get('invalid_moves', 0)),
                'params': json.dumps(params, separators=(',', ':')),
            })
        finally:
            if str(root_dir) in sys.path:
                sys.path.remove(str(root_dir))

print('total rows:', len(rows))
print('versions:', sorted({row['version'] for row in rows}))


In [ ]:
# 7. Coleta e persistência das métricas em CSV
# Coletar, quando disponíveis, cenário, semente, versão experimental, problema resolvido, motivo da parada, gerações executadas, tempo de execução, fitness do melhor indivíduo, distância de Manhattan final, comprimento do melhor cromossomo e movimentos inválidos. Salvar os resultados em um novo CSV, por exemplo results/normalization_ab_experiment.csv, sem sobrescrever arquivos históricos e sem inventar valores que não tenham sido medidos.

fieldnames = [
    'version', 'scenario', 'seed', 'size', 'initial_state', 'solved', 'stop_reason', 'generations_run',
    'time_total_seconds', 'best_fitness', 'best_manhattan_distance', 'best_chromosome_length',
    'invalid_moves', 'params'
]
with OUT_CSV.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)
print('CSV written:', OUT_CSV)
print('rows written:', len(rows))


In [ ]:
# 8. Análise comparativa de resolução e desempenho
# Comparar as variantes considerando taxa de resolução geral e por cenário, diferença absoluta na taxa de resolução, tempo de execução, gerações executadas, comprimento dos cromossomos, distância de Manhattan final e potenciais regressões. Diferenciar claramente entre redução de redundâncias, melhoria na fitness e melhoria real na taxa de resolução, evitando declarar uma variante superior com base em uma única métrica.

import pandas as pd

if OUT_CSV.exists():
    df = pd.read_csv(OUT_CSV)
    summary = df.groupby('version').agg(
        runs=('scenario', 'size'),
        solved=('solved', 'sum'),
        rate=('solved', lambda s: s.mean()),
        avg_time=('time_total_seconds', 'mean'),
        avg_generations=('generations_run', 'mean'),
        avg_manhattan=('best_manhattan_distance', 'mean'),
        avg_length=('best_chromosome_length', 'mean'),
    )
    print(summary)
else:
    print('CSV not generated yet; execute the notebook cells in order.')


In [ ]:
# 9. Validação final e verificação da integridade do repositório
# Executar a suíte completa de testes na implementação principal para confirmar que o experimento não impactou o código em produção. Verificar que nenhum arquivo principal foi modificado, listar os comandos utilizados, registrar o número de execuções e o caminho do CSV de resultados, e documentar limitações que possam afetar a validade da comparação, como diferenças entre as variantes além da normalização e qualquer viés de amostragem ou dependência externa.

print('Validation notebook ready. Run all cells in order to perform the controlled A/B comparison and generate the CSV result.')
print('Main files preserved: genetic_algorithm/operators.py, genetic_algorithm/solver.py, genetic_algorithm/fitness.py, genetic_algorithm/individual.py, puzzle/board.py')
